# CognIA — Resultados finales RF-based v12

Notebook ejecutivo/técnico para presentar los resultados finales de la campaña `hybrid_final_rf_plus_maximize_metrics_v1`.

**Objetivo del notebook**

- Mostrar resultados finales v12 con gráficas claras y detalladas.
- Comparar v11 → v12 y, si está disponible, v10 → v11 → v12.
- Evidenciar métricas, guardrails, deltas, clasificación operacional, estabilidad, calibración y riesgos.
- Mantener visualizaciones sobrias, legibles y reproducibles.
- Priorizar F1 y recall, sin ocultar tradeoffs de precision, BA, Brier o robustez.

> Este notebook no reentrena modelos. Solo lee artifacts versionados del repo.


In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Estilo visual sobrio y consistente.
plt.rcParams.update({
    "figure.figsize": (13, 6.5),
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "api").exists():
            return p
    raise RuntimeError("No pude detectar la raíz del repo. Ejecuta el notebook desde CognIA o ajusta REPO_ROOT manualmente.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)

LINE = "hybrid_final_rf_plus_maximize_metrics_v1"
FREEZE = "v12"

DATA_DIR = REPO_ROOT / "data" / LINE
ACTIVE_DIR = REPO_ROOT / f"data/hybrid_active_modes_freeze_{FREEZE}"
OP_DIR = REPO_ROOT / f"data/hybrid_operational_freeze_{FREEZE}"
REPORT_DIR = DATA_DIR / "notebook_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "comparison_v11_v12": DATA_DIR / "tables/final_v11_vs_v12_comparison.csv",
    "reference_v10_v11_v12": DATA_DIR / "tables/reference_v10_vs_v11_vs_v12_comparison.csv",
    "selected_champions": DATA_DIR / "tables/selected_rf_champions_v12.csv",
    "active_models": ACTIVE_DIR / "tables/hybrid_active_models_30_modes.csv",
    "operational_champions": OP_DIR / "tables/hybrid_operational_final_champions.csv",
    "integrity_validator": DATA_DIR / "validation/final_campaign_integrity_validator_v12.json",
    "rf_only_validator": DATA_DIR / "validation/rf_only_validator.csv",
    "feature_compat": DATA_DIR / "validation/feature_columns_compatibility_validator.csv",
    "questionnaire_unchanged": DATA_DIR / "validation/questionnaire_unchanged_validator.csv",
    "supabase_sync": DATA_DIR / "supabase_sync/supabase_sync_verification_v12.json",
    "bootstrap": DATA_DIR / "validation/selected_rf_bootstrap_audit.csv",
    "stress": DATA_DIR / "validation/selected_rf_stress_audit.csv",
    "importance": DATA_DIR / "validation/selected_rf_feature_importance.csv",
    "elimination_similarity": DATA_DIR / "validation/final_elimination_prediction_similarity.csv",
}

for name, path in paths.items():
    print(f"{name:28s}", "OK" if path.exists() else "missing", path.relative_to(REPO_ROOT) if path.exists() else path)


## 1. Carga robusta de artifacts

El notebook está preparado para no fallar si algún archivo opcional no existe.  
Los archivos críticos para la presentación son:

- `final_v11_vs_v12_comparison.csv`
- `hybrid_active_models_30_modes.csv`
- `hybrid_operational_final_champions.csv`
- validadores de integridad / RF-only / Supabase, si están disponibles.


In [ ]:
def read_csv_optional(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"[WARN] No existe: {path.relative_to(REPO_ROOT) if path.is_relative_to(REPO_ROOT) else path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)

def read_json_optional(path):
    path = Path(path)
    if not path.exists():
        print(f"[WARN] No existe: {path.relative_to(REPO_ROOT) if path.is_relative_to(REPO_ROOT) else path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

comparison = read_csv_optional(paths["comparison_v11_v12"])
reference = read_csv_optional(paths["reference_v10_v11_v12"])
selected = read_csv_optional(paths["selected_champions"])
active = read_csv_optional(paths["active_models"])
operational = read_csv_optional(paths["operational_champions"])
rf_only = read_csv_optional(paths["rf_only_validator"])
feature_compat = read_csv_optional(paths["feature_compat"])
questionnaire_unchanged = read_csv_optional(paths["questionnaire_unchanged"])
bootstrap = read_csv_optional(paths["bootstrap"])
stress = read_csv_optional(paths["stress"])
importance = read_csv_optional(paths["importance"])
elim_similarity = read_csv_optional(paths["elimination_similarity"])

integrity = read_json_optional(paths["integrity_validator"])
supabase = read_json_optional(paths["supabase_sync"])

print("comparison:", comparison.shape)
print("selected:", selected.shape)
print("active:", active.shape)
print("operational:", operational.shape)


In [ ]:
# Normalización mínima de etiquetas para visualizaciones.
if not comparison.empty:
    comparison["slot"] = comparison["domain"].astype(str) + "/" + comparison["role"].astype(str) + "/" + comparison["mode"].astype(str).str.replace("caregiver_", "", regex=False).str.replace("psychologist_", "", regex=False)
    comparison["role_mode"] = comparison["role"].astype(str) + "/" + comparison["mode"].astype(str)
    comparison["mode_short"] = comparison["mode"].astype(str).str.replace("caregiver_", "", regex=False).str.replace("psychologist_", "", regex=False)
    comparison["domain_role"] = comparison["domain"].astype(str) + "/" + comparison["role"].astype(str)

metric_cols = ["f1", "recall", "precision", "specificity", "balanced_accuracy", "roc_auc", "pr_auc", "brier"]
delta_cols = [f"delta_{m}" for m in metric_cols if f"delta_{m}" in comparison.columns]
old_cols = [f"old_{m}" for m in metric_cols if f"old_{m}" in comparison.columns]
new_cols = [f"new_{m}" for m in metric_cols if f"new_{m}" in comparison.columns]

display(comparison.head(3))


## 2. Resumen ejecutivo cuantitativo

Esta sección resume:

- cantidad de slots activos,
- cumplimiento RF-based,
- guardrails duros,
- deltas agregados v11 → v12,
- distribución operacional.


In [ ]:
def fmt_pct(x, digits=1):
    if pd.isna(x):
        return "—"
    return f"{100*x:.{digits}f}%"

def fmt_num(x, digits=4):
    if pd.isna(x):
        return "—"
    return f"{x:.{digits}f}"

summary_rows = []

if not comparison.empty:
    summary_rows.extend([
        {"indicador": "Slots comparados", "valor": len(comparison)},
        {"indicador": "F1 medio v11", "valor": fmt_num(comparison["old_f1"].mean())},
        {"indicador": "F1 medio v12", "valor": fmt_num(comparison["new_f1"].mean())},
        {"indicador": "Δ F1 medio", "valor": fmt_num(comparison["delta_f1"].mean())},
        {"indicador": "Δ Recall medio", "valor": fmt_num(comparison["delta_recall"].mean())},
        {"indicador": "Δ Precision media", "valor": fmt_num(comparison["delta_precision"].mean())},
        {"indicador": "Δ BA media", "valor": fmt_num(comparison["delta_balanced_accuracy"].mean())},
        {"indicador": "Δ Brier medio", "valor": fmt_num(comparison["delta_brier"].mean())},
        {"indicador": "Slots con F1 mejorado", "valor": int((comparison["delta_f1"] > 1e-9).sum())},
        {"indicador": "Slots con F1 empatado", "valor": int((comparison["delta_f1"].abs() <= 1e-9).sum())},
        {"indicador": "Slots con F1 menor", "valor": int((comparison["delta_f1"] < -1e-9).sum())},
    ])

if not active.empty:
    summary_rows.extend([
        {"indicador": "Modelos activos finales", "valor": len(active)},
        {"indicador": "Familias finales únicas", "valor": ", ".join(sorted(active["model_family"].dropna().astype(str).unique())) if "model_family" in active else "—"},
    ])
    for m in ["recall", "specificity", "roc_auc", "pr_auc"]:
        if m in active:
            summary_rows.append({"indicador": f"Máximo {m}", "valor": fmt_num(active[m].max())})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if not operational.empty and "final_operational_class" in operational.columns:
    display(operational["final_operational_class"].value_counts().rename_axis("final_operational_class").reset_index(name="n"))
elif not active.empty and "final_operational_class" in active.columns:
    display(active["final_operational_class"].value_counts().rename_axis("final_operational_class").reset_index(name="n"))


## 3. Gráfica ejecutiva: delta agregado v11 → v12

Interpretación:

- Valores positivos en F1, recall, precision y BA son mejora.
- En Brier, valores negativos son mejora porque menor Brier indica mejor calibración/probabilidad.


In [ ]:
def savefig(name):
    path = REPORT_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print("saved:", path.relative_to(REPO_ROOT))

if not comparison.empty:
    agg = pd.Series({
        "F1": comparison["delta_f1"].mean(),
        "Recall": comparison["delta_recall"].mean(),
        "Precision": comparison["delta_precision"].mean(),
        "BA": comparison["delta_balanced_accuracy"].mean(),
        "Brier": comparison["delta_brier"].mean(),
    })
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(agg))
    bars = ax.bar(x, agg.values, color=["#4C78A8" if v >= 0 else "#A0A0A0" for v in agg.values])
    ax.axhline(0, color="#333333", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(agg.index)
    ax.set_title("Delta promedio v11 → v12 por métrica")
    ax.set_ylabel("Delta promedio")
    for bar, value in zip(bars, agg.values):
        ax.text(bar.get_x() + bar.get_width()/2, value + (0.0008 if value >= 0 else -0.0008), f"{value:+.4f}",
                ha="center", va="bottom" if value >= 0 else "top")
    savefig("01_delta_promedio_v11_v12.png")
    plt.show()


## 4. Vista completa de métricas finales por slot

Tabla ordenada por dominio/rol/modo con métricas finales v12 y deltas frente a v11.  
Útil para QA y revisión granular.


In [ ]:
if not comparison.empty:
    cols = [
        "domain", "role", "mode",
        "old_f1", "new_f1", "delta_f1",
        "old_recall", "new_recall", "delta_recall",
        "old_precision", "new_precision", "delta_precision",
        "old_balanced_accuracy", "new_balanced_accuracy", "delta_balanced_accuracy",
        "old_brier", "new_brier", "delta_brier",
        "old_class", "new_class", "selection_reason"
    ]
    cols = [c for c in cols if c in comparison.columns]
    styled = comparison[cols].sort_values(["domain", "role", "mode"]).style.format({
        c: "{:.4f}" for c in cols if c.startswith(("old_", "new_", "delta_"))
    })
    display(styled)


## 5. F1 antes/después por slot

Gráfica horizontal para comparar F1 v11 vs v12 en los 30 slots.  
Se ordena por delta F1 para ver rápidamente dónde hubo más ganancia, empate o regresión.


In [ ]:
if not comparison.empty:
    df = comparison.sort_values("delta_f1", ascending=True).reset_index(drop=True)
    y = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(13, 12))
    ax.hlines(y, df["old_f1"], df["new_f1"], color="#B0B0B0", linewidth=2)
    ax.scatter(df["old_f1"], y, label="v11", s=36, color="#9E9E9E")
    ax.scatter(df["new_f1"], y, label="v12", s=42, color="#4C78A8")
    ax.set_yticks(y)
    ax.set_yticklabels(df["slot"])
    ax.set_xlabel("F1")
    ax.set_title("F1 por slot: v11 vs v12")
    ax.legend(loc="lower right")
    for i, r in df.iterrows():
        ax.text(max(r["old_f1"], r["new_f1"]) + 0.003, i, f"{r['delta_f1']:+.3f}", va="center", fontsize=8)
    savefig("02_f1_v11_vs_v12_por_slot.png")
    plt.show()
